In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/sample.csv.csv
/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/train.csv
/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/test.csv


Import Libraries

In [2]:
from sklearn.linear_model import LinearRegression,SGDRegressor,Lasso,Ridge,LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,PolynomialFeatures,MinMaxScaler,LabelEncoder,StandardScaler,MaxAbsScaler,FunctionTransformer
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor,BaggingRegressor,RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score,train_test_split
from scipy.stats import pearsonr
import seaborn as sns

Get the Training dataset

In [3]:
train=pd.read_csv('/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/train.csv')
train

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,extra,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2023-06-28 17:20:21,2023-06-28 16:34:45,1.0,2.14,1.0,N,120,9,Credit Card,2.5,7.165589,0.0,1.0,20.64,2.5,0.00
1,0,2023-06-29 23:05:01,2023-06-29 22:01:35,1.0,2.70,1.0,N,15,215,Credit Card,3.5,6.067401,0.0,1.0,25.55,2.5,0.00
2,1,2023-06-30 10:19:31,2023-06-30 11:13:10,1.0,1.15,1.0,N,167,223,Credit Card,0.0,4.111547,0.0,1.0,17.64,2.5,0.00
3,0,2023-06-29 13:23:09,2023-06-29 14:20:01,1.0,0.40,1.0,N,128,239,Credit Card,2.5,6.411079,0.0,1.0,12.80,2.5,0.00
4,1,2023-06-29 22:03:32,2023-06-29 22:22:22,3.0,1.10,1.0,N,203,52,Credit Card,1.0,4.769377,0.0,1.0,18.00,2.5,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174995,1,2023-06-30 22:50:57,2023-06-30 22:22:22,3.0,3.45,1.0,N,147,167,Credit Card,1.0,8.732495,0.0,1.0,28.08,2.5,0.00
174996,1,2023-06-30 13:03:33,2023-06-30 14:04:57,1.0,9.44,1.0,N,154,191,Cash,5.0,0.283275,0.0,1.0,59.95,2.5,1.75
174997,0,2023-06-29 11:03:32,2023-06-29 12:13:34,1.0,2.40,1.0,N,168,106,Credit Card,2.5,4.245354,0.0,1.0,33.50,2.5,0.00
174998,1,2023-06-29 19:47:17,2023-06-29 19:08:55,1.0,4.71,1.0,N,240,100,Credit Card,2.5,10.479776,0.0,1.0,40.80,2.5,0.00


In [4]:
a,_=pearsonr(np.array(train['PULocationID']),np.array(train['total_amount']))
a

0.0007150353472589551

Separate the features from target variable

In [5]:
labels=train['total_amount']
feature=train.drop(['total_amount'],axis=1)

Split the training dataset to get train and validation dataset by using train test split

In [6]:
train,val,y_train_clean,y_val_clean=train_test_split(feature,labels,test_size=0.1,random_state=42)

statistical summary of the training dataset

In [7]:
train.describe()

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,extra,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,Airport_fee
count,157500.000000,152041.000000,157500.000000,152041.000000,157500.000000,157500.000000,157500.000000,157500.000000,157500.000000,157500.000000,152041.000000,152041.000000
mean,0.728267,1.357969,5.308771,1.519380,132.736806,132.747073,1.931891,6.127055,0.647175,0.979703,2.247946,0.158275
std,0.445739,0.891861,416.332666,6.523127,76.201345,76.218390,1.948683,4.625129,2.328232,0.198644,0.817551,0.511272
min,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,-7.500000,0.000079,-29.300000,-1.000000,-2.500000,-1.750000
25%,0.000000,1.000000,1.080000,1.000000,67.000000,67.000000,0.000000,3.471385,0.000000,1.000000,2.500000,0.000000
50%,1.000000,1.000000,1.840000,1.000000,133.000000,133.000000,1.000000,5.288915,0.000000,1.000000,2.500000,0.000000
75%,1.000000,1.000000,3.610000,1.000000,199.000000,199.000000,2.500000,7.505016,0.000000,1.000000,2.500000,0.000000
max,2.000000,9.000000,135182.060000,99.000000,264.000000,264.000000,11.750000,484.876151,80.000000,1.000000,2.500000,1.750000


Statistical summary of the validation set

In [8]:
val.describe()

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,extra,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,Airport_fee
count,17500.000000,16882.000000,17500.000000,16882.000000,17500.000000,17500.000000,17500.000000,17500.000000,17500.000000,17500.000000,16882.000000,16882.000000
mean,0.729371,1.355053,3.680365,1.508648,132.472229,132.290629,1.934414,6.131474,0.643592,0.979554,2.238183,0.163784
std,0.444426,0.886077,4.891255,6.438268,75.675986,75.959959,1.946880,4.480249,2.328714,0.199955,0.834039,0.518189
min,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,-7.500000,0.000713,-26.550000,-1.000000,-2.500000,-1.750000
25%,0.000000,1.000000,1.070000,1.000000,67.000000,67.000000,0.000000,3.494520,0.000000,1.000000,2.500000,0.000000
50%,1.000000,1.000000,1.830000,1.000000,132.000000,133.000000,1.750000,5.262071,0.000000,1.000000,2.500000,0.000000
75%,1.000000,1.000000,3.660000,1.000000,198.000000,198.000000,2.500000,7.472769,0.000000,1.000000,2.500000,0.000000
max,2.000000,6.000000,71.940000,99.000000,264.000000,264.000000,11.750000,84.032617,36.050000,1.000000,2.500000,1.750000


get the test data

In [9]:
test=pd.read_csv('/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/test.csv')

Statistical summary of the test data

In [10]:
test.describe()

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,extra,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,Airport_fee
count,50000.000000,48221.000000,50000.000000,48221.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,48221.000000,48221.000000
mean,0.730280,1.358309,3.999013,1.567014,132.208160,132.559300,1.918050,6.107765,0.615867,0.981354,2.255345,0.152133
std,0.444584,0.879948,78.958759,6.875115,76.483766,76.410602,1.938568,4.408572,2.289421,0.190203,0.803190,0.502866
min,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,-7.500000,0.000409,-23.000000,-1.000000,-2.500000,-1.750000
25%,0.000000,1.000000,1.090000,1.000000,66.000000,67.000000,0.000000,3.464018,0.000000,1.000000,2.500000,0.000000
50%,1.000000,1.000000,1.850000,1.000000,132.000000,133.000000,1.000000,5.271687,0.000000,1.000000,2.500000,0.000000
75%,1.000000,1.000000,3.600000,1.000000,199.000000,199.000000,2.500000,7.504048,0.000000,1.000000,2.500000,0.000000
max,2.000000,8.000000,17624.430000,99.000000,264.000000,264.000000,11.750000,96.551343,47.750000,1.000000,2.500000,1.750000


Drop unwanted columns

In [11]:
train=train.drop(['PULocationID'],axis=1)
test=test.drop(['PULocationID'],axis=1)
val=val.drop(['PULocationID'],axis=1)
test=test.drop(['DOLocationID'],axis=1)
train=train.drop(['DOLocationID'],axis=1)
val=val.drop(['DOLocationID'],axis=1)
test=test.drop(['store_and_fwd_flag'],axis=1)
train=train.drop(['store_and_fwd_flag'],axis=1)
val=val.drop(['store_and_fwd_flag'],axis=1)

Find missing values by using KNNImputer

In [12]:
pre=[('knn',KNNImputer(n_neighbors=2, weights="uniform"),['passenger_count','RatecodeID','congestion_surcharge','Airport_fee'])]
ct=ColumnTransformer(transformers=pre, remainder='passthrough', verbose_feature_names_out=False)




In [13]:
train_clean=pd.DataFrame(ct.fit_transform(train),columns=ct.get_feature_names_out())


In [14]:
val_clean=pd.DataFrame(ct.transform(val),columns=ct.get_feature_names_out())


In [15]:
test_clean=pd.DataFrame(ct.transform(test),columns=ct.get_feature_names_out())


Cluster the columns "tpep_dropoff_datetime and tpep_pickup_datetime" into four groups (Group 1, Group 2, Group 3 and Group 4)

In [16]:
train_clean['tpep_dropoff_datetime']=pd.to_datetime(train_clean['tpep_dropoff_datetime'])
test_clean['tpep_dropoff_datetime']=pd.to_datetime(test_clean['tpep_dropoff_datetime'])
train_clean['tpep_pickup_datetime']=pd.to_datetime(train_clean['tpep_pickup_datetime'])
test_clean['tpep_pickup_datetime']=pd.to_datetime(test_clean['tpep_pickup_datetime'])
val_clean['tpep_dropoff_datetime']=pd.to_datetime(val_clean['tpep_dropoff_datetime'])
val_clean['tpep_pickup_datetime']=pd.to_datetime(val_clean['tpep_pickup_datetime'])



In [17]:
train_clean['d_time_only']=train_clean.tpep_dropoff_datetime.dt.time
train_clean['p_time_only']=train_clean.tpep_pickup_datetime.dt.time

In [18]:
val_clean['d_time_only']=val_clean.tpep_dropoff_datetime.dt.time
val_clean['p_time_only']=val_clean.tpep_pickup_datetime.dt.time

In [19]:
test_clean['d_time_only']=test_clean.tpep_dropoff_datetime.dt.time
test_clean['p_time_only']=test_clean.tpep_pickup_datetime.dt.time


In [20]:
train_clean=train_clean.drop(['tpep_dropoff_datetime','tpep_pickup_datetime'],axis=1)


In [21]:
val_clean=val_clean.drop(['tpep_dropoff_datetime','tpep_pickup_datetime'],axis=1)


In [22]:
test_clean=test_clean.drop(['tpep_dropoff_datetime','tpep_pickup_datetime'],axis=1)

In [23]:
i=0
for x in train_clean['p_time_only']:
  train_clean['p_time_only'][i]=str(train_clean['p_time_only'][i])
  train_clean['p_time_only'][i]=int(train_clean['p_time_only'][i][:2])
  
  i=i+1


In [24]:
i=0
c=[]
l=[]
d={}
m=[]
for x in train_clean['p_time_only'].value_counts():
  a=(i,train_clean['p_time_only'].value_counts()[i])
  (list(a))
  labe=a[0]
  d[a[0]]=a[1]
  m.append(int(a[1]))
  c.append(list(a))
  l.append(labe)
  i=i+1
m.sort()
d

{0: 4564,
 1: 2211,
 2: 1274,
 3: 843,
 4: 777,
 5: 1150,
 6: 2177,
 7: 3602,
 8: 4898,
 9: 5563,
 10: 5943,
 11: 6491,
 12: 6886,
 13: 7225,
 14: 7703,
 15: 8052,
 16: 9960,
 17: 12509,
 18: 13311,
 19: 12419,
 20: 10996,
 21: 10772,
 22: 10168,
 23: 8006}

In [25]:
L=[]
for i in range(len(m)):
   for x in d:
    if d[x]==m[i]:
        new=list((x,m[i],(i//6)+1))
        L.append(new)
        i=i+1
        break
D={}
for i in range(len(L)):
    D[L[i][0]]=L[i][2]
D

{4: 1,
 3: 1,
 5: 1,
 2: 1,
 6: 1,
 1: 1,
 7: 2,
 0: 2,
 8: 2,
 9: 2,
 10: 2,
 11: 2,
 12: 3,
 13: 3,
 14: 3,
 23: 3,
 15: 3,
 16: 3,
 22: 4,
 21: 4,
 20: 4,
 19: 4,
 17: 4,
 18: 4}

In [26]:
k=0  
for x in train_clean['p_time_only']:
    
          train_clean['p_time_only'][k]=D[x]
          
          k=k+1
          
    

In [27]:
i=0
for x in train_clean['d_time_only']:
  train_clean['d_time_only'][i]=str(train_clean['d_time_only'][i])
  train_clean['d_time_only'][i]=int(train_clean['d_time_only'][i][:2])
  i=i+1

   

In [28]:
i=0
c=[]
l=[]
d={}
m=[]
for x in train_clean['d_time_only'].value_counts():
  a=(i,train_clean['d_time_only'].value_counts()[i])
  labe=a[0]
  d[a[0]]=a[1]
  m.append(int(a[1]))
  c.append(list(a))
  l.append(labe)
  i=i+1
m.sort()


In [29]:
L=[]
for i in range(len(m)):
   for x in d:
    if d[x]==m[i]:
        new=list((x,m[i],(i//6)+1))
        L.append(new)
        i=i+1
        break
D={}
for i in range(len(L)):
    D[L[i][0]]=L[i][2]


In [30]:
k=0  
for x in train_clean['d_time_only']:
    
          train_clean['d_time_only'][k]=D[x]
          
          k=k+1
          
    
        
 
        

In [31]:
i=0
for x in val_clean['p_time_only']:
  val_clean['p_time_only'][i]=str(val_clean['p_time_only'][i])
  val_clean['p_time_only'][i]=int(val_clean['p_time_only'][i][:2])
  
  i=i+1


In [32]:
i=0
c=[]
l=[]
d={}
m=[]
for x in val_clean['p_time_only'].value_counts():
  a=(i,val_clean['p_time_only'].value_counts()[i])
  print(list(a))
  labe=a[0]
  d[a[0]]=a[1]
  m.append(int(a[1]))
  c.append(list(a))
  l.append(labe)
  i=i+1
m.sort()

[0, 474]
[1, 249]
[2, 142]
[3, 101]
[4, 101]
[5, 122]
[6, 237]
[7, 419]
[8, 542]
[9, 620]
[10, 687]
[11, 694]
[12, 754]
[13, 802]
[14, 873]
[15, 910]
[16, 1148]
[17, 1411]
[18, 1487]
[19, 1405]
[20, 1158]
[21, 1171]
[22, 1091]
[23, 902]


In [33]:
L=[]
for i in range(len(m)):
   
   for x in d:
    if d[x]==m[i]:
        new=list((x,m[i],(i//6)+1))
        L.append(new)
        i=i+1
        d[x]=''
        break
D={}
for i in range(len(L)):
    D[L[i][0]]=L[i][2]


In [34]:
k=0  
for x in val_clean['p_time_only']:
    
          val_clean['p_time_only'][k]=D[x]
          
          k=k+1
          
    
        
  
        
    

In [35]:
i=0
for x in val_clean['d_time_only']:
  val_clean['d_time_only'][i]=str(val_clean['d_time_only'][i])
  val_clean['d_time_only'][i]=int(val_clean['d_time_only'][i][:2])
  i=i+1

   

In [36]:
i=0
c=[]
l=[]
d={}
m=[]
for x in val_clean['d_time_only'].value_counts():
  a=(i,val_clean['d_time_only'].value_counts()[i])
  labe=a[0]
  d[a[0]]=a[1]
  m.append(int(a[1]))
  c.append(list(a))
  l.append(labe)
  i=i+1
m.sort()


In [37]:
L=[]
for i in range(len(m)):
   for x in d:
    if d[x]==m[i]:
        new=list((x,m[i],(i//6)+1))
        L.append(new)
        i=i+1
        break
D={}
for i in range(len(L)):
    D[L[i][0]]=L[i][2]


In [38]:
k=0  
for x in val_clean['d_time_only']:
    
          val_clean['d_time_only'][k]=D[x]
          
          k=k+1
          
    
        
 
        

In [39]:
i=0
for x in test_clean['p_time_only']:
  test_clean['p_time_only'][i]=str(test_clean['p_time_only'][i])
  test_clean['p_time_only'][i]=int(test_clean['p_time_only'][i][:2])
  i=i+1

In [40]:
i=0
c=[]
l=[]
d={}
m=[]
for x in test_clean['p_time_only'].value_counts():
  a=(i,test_clean['p_time_only'].value_counts()[i])
  labe=a[0]
  d[a[0]]=a[1]
  m.append(int(a[1]))
  c.append(list(a))
  l.append(labe)
  i=i+1
m.sort()


In [41]:
L=[]
for i in range(len(m)):
   for x in d:
    if d[x]==m[i]:
        new=list((x,m[i],(i//6)+1))
        L.append(new)
        i=i+1
        break
D={}
for i in range(len(L)):
    D[L[i][0]]=L[i][2]


In [42]:
k=0  
for x in test_clean['p_time_only']:
    
          test_clean['p_time_only'][k]=D[x]
          
          k=k+1   
        

In [43]:
i=0
for x in test_clean['d_time_only']:
  test_clean['d_time_only'][i]=str(test_clean['d_time_only'][i])
  test_clean['d_time_only'][i]=int(test_clean['d_time_only'][i][:2])
  i=i+1



In [44]:
i=0
c=[]
l=[]
d={}
m=[]
for x in test_clean['d_time_only'].value_counts():
  a=(i,test_clean['d_time_only'].value_counts()[i])
  labe=a[0]
  d[a[0]]=a[1]
  m.append(int(a[1]))
  c.append(list(a))
  l.append(labe)
  i=i+1
m.sort()

In [45]:
L=[]
for i in range(len(m)):
   for x in d:
    if d[x]==m[i]:
        new=list((x,m[i],(i//6)+1))
        L.append(new)
        i=i+1
        break
D={}
for i in range(len(L)):
    D[L[i][0]]=L[i][2]


In [46]:
k=0  
for x in test_clean['d_time_only']:
    
          test_clean['d_time_only'][k]=D[x]
          
          k=k+1   
        

Encode nominal categorical variables into numerical values

In [47]:
trf=[('ohe',OneHotEncoder(),['VendorID','payment_type'])]
ct2=ColumnTransformer(transformers=trf, remainder='passthrough')


In [48]:
train_clean=pd.DataFrame(ct2.fit_transform(train_clean),columns=ct2.get_feature_names_out())

In [49]:
val_clean=pd.DataFrame(ct2.transform(val_clean),columns=ct2.get_feature_names_out())

In [50]:
X_test_clean=pd.DataFrame(ct2.transform(test_clean),columns=ct2.get_feature_names_out())

Encode ordinal categorical values 

In [51]:
c=LabelEncoder()
train_clean['remainder__d_time_only']=c.fit_transform(train_clean['remainder__d_time_only'])
val_clean['remainder__d_time_only']=c.transform(val_clean['remainder__d_time_only'])
X_test_clean['remainder__d_time_only']=c.transform(X_test_clean['remainder__d_time_only'])


In [52]:
c=LabelEncoder()
train_clean['remainder__p_time_only']=c.fit_transform(train_clean['remainder__p_time_only'])
val_clean['remainder__p_time_only']=c.transform(val_clean['remainder__p_time_only'])
X_test_clean['remainder__p_time_only']=c.transform(X_test_clean['remainder__p_time_only'])


In [53]:
c=LabelEncoder()
train_clean['remainder__RatecodeID']=c.fit_transform(train_clean['remainder__RatecodeID'])
val_clean['remainder__RatecodeID']=c.fit_transform(val_clean['remainder__RatecodeID'])
X_test_clean['remainder__RatecodeID']=c.fit_transform(X_test_clean['remainder__RatecodeID'])


In [54]:
c=LabelEncoder()
train_clean['remainder__passenger_count']=c.fit_transform(train_clean['remainder__passenger_count'])
val_clean['remainder__passenger_count']=c.fit_transform(val_clean['remainder__passenger_count'])
X_test_clean['remainder__passenger_count']=c.fit_transform(X_test_clean['remainder__passenger_count'])


Scale the entire dataset by using standard scaler

In [55]:

ss=StandardScaler()

In [56]:
a=pd.DataFrame(ss.fit_transform(train_clean))
a_val=pd.DataFrame(ss.transform(val_clean))
b=pd.DataFrame(ss.transform(X_test_clean))

In [57]:
train_clean=pd.DataFrame(a)
val_clean=pd.DataFrame(a_val)
X_test_clean=pd.DataFrame(b)

Tune the hyper parameter value for decision tree regressor model

In [58]:
dtr=DecisionTreeRegressor()
dtr_grid=GridSearchCV(dtr,param_grid={'max_depth':[5,15,25,35]})
dtr_grid.fit(train_clean,y_train_clean)
dtr_grid.best_params_

{'max_depth': 15}

Apply Decision Tree Regressor as an estimator for Bagging Regressor model with best parameter

In [59]:
dtr=DecisionTreeRegressor(max_depth=15,random_state=42)
bgr=BaggingRegressor(dtr,n_estimators=100,random_state=42)
bgr.fit(train_clean,y_train_clean)
pred=bgr.predict(train_clean)

In [60]:
r2_score(y_train_clean,pred)

0.9763924496182015

In [61]:
pred=bgr.predict(val_clean)
r2_score(y_val_clean,pred)

0.9611543547203814

In [62]:
pred=bgr.predict(X_test_clean)

In [63]:
submission=pd.DataFrame(columns=['ID','total_amount'])
submission['ID']=[i for i in range(1,len(pred)+1)]
submission['total_amount']=pred
submission.to_csv('submission.csv', index=False)
